# Build decoy augmentation training data

Selects, for every training ligand, the 200 most similar molecules from the curated
decoy library that still sit below a Tanimoto of 0.5 - the ligand-specific decoy
enhancement strategy of Fig. 1c. Writes both `decoy_fp2` (the fingerprints the model
trains on) and `decoy_smiles` (the molecules themselves).

Needs `decoy_library.smi` (92,626 entries) and `data/hariboss/hariboss_10fd_with_decoys.pkl`,
both from the Zenodo record linked in the repository README.

Runtime: a few minutes (92,626 fingerprints, then one vectorized search per distinct ligand).

In [ ]:
import sys
sys.path.append("..")
import pickle
from smartbind.preprocess import convert_smiles_to_pf2
from smartbind.decoy import build_fp_matrix, top_decoy_indices
import tqdm

In [ ]:
TOP_NUM = 200
TANIMOTO_THRESHOLD = 0.5

# read the smi file
with open('decoy_library.smi', 'r') as f:
    decoy_smiles = [line.strip().split()[0] for line in f if line.strip()]
print(f"Total decoys loaded: {len(decoy_smiles)}")

# drop duplicated SMILES, keeping the original order
decoy_smiles = list(dict.fromkeys(decoy_smiles))
print(f"Unique decoys: {len(decoy_smiles)}")

In [ ]:
# keep smiles and fingerprints aligned: an unparsable SMILES drops both
valid_smiles, decoy_smiles_fp2 = [], []
for smi in tqdm.tqdm(decoy_smiles, desc="fingerprinting decoys"):
    try:
        fp = convert_smiles_to_pf2(smi)
    except Exception:
        continue
    if sum(fp) == 0:  # empty fingerprint, similarity is undefined
        continue
    valid_smiles.append(smi)
    decoy_smiles_fp2.append(fp)
decoy_smiles = valid_smiles
print(f"Decoys with a valid FP2: {len(decoy_smiles)}")
decoy_matrix = build_fp_matrix(decoy_smiles_fp2)

In [ ]:
with open('../data/hariboss/hariboss_10fd_with_decoys.pkl', 'rb') as f:
    training_data = pickle.load(f)
print(f"Total training PDBs loaded: {len(training_data)}")

cache = {}  # the same ligand shows up in many complexes, only search once
min_decoys = TOP_NUM
for k, v in tqdm.tqdm(training_data.items()):
    for kk, vv in v.items():
        fp2_str = ''.join(map(str, vv['fp2']))
        if fp2_str not in cache:
            cache[fp2_str] = top_decoy_indices(decoy_matrix, fp2_str,
                                               top_num=TOP_NUM,
                                               tanimoto_threshold=TANIMOTO_THRESHOLD)
        idx = cache[fp2_str]
        vv['decoy_fp2'] = [decoy_smiles_fp2[i] for i in idx]
        vv['decoy_smiles'] = [decoy_smiles[i] for i in idx]
        min_decoys = min(min_decoys, len(idx))

print(f"Unique ligand fingerprints searched: {len(cache)}")
print(f"Fewest decoys found for a single ligand: {min_decoys}")

In [ ]:
# save the augmented data
with open('../data/hariboss/hariboss_10fd_with_decoys_rebuilt.pkl', 'wb') as f:
    pickle.dump(training_data, f)
print("Augmented training data saved with decoy fingerprints and smiles.")